Dhay Alfozan - dhaykh.2006@gmail.com

**Project Objective**
Analyze 2025 sales data across 6 stores in 4 cities to understand revenue trends, category performance, in-store vs. online differences, and return/satisfaction patterns — supporting pricing, inventory, and expansion decisions

In [94]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df = pd.read_csv("retail_sales.csv", parse_dates=["date"])
df.head()

,date,store,city,category,channel,units_sold,revenue_sar,returns_units,foot_traffic,csat
0,2025-01-01,Riyadh-Olaya,Riyadh,Electronics,In-Store,52,45097.25,1,950.0,3.98
1,2025-01-01,Riyadh-Olaya,Riyadh,Electronics,Online,12,9323.66,0,NaN,4.00
2,2025-01-01,Riyadh-Olaya,Riyadh,Apparel,In-Store,57,7770.91,2,950.0,4.37
3,2025-01-01,Riyadh-Olaya,Riyadh,Apparel,Online,7,909.04,0,NaN,4.03
4,2025-01-01,Riyadh-Olaya,Riyadh,Home & Kitchen,In-Store,52,11611.75,2,950.0,3.75


In [95]:
import pandas as pd

df = pd.read_csv('retail_sales.csv', parse_dates=['date'])
df['month'] = df['date'].values.astype('datetime64[M]')

latest_month = df['month'].max()
last12 = df[df['month'] >= latest_month - pd.DateOffset(months=11)]

# 1) National average CSAT (latest month) — replaces "digital adoption"
national_csat = round(df[df['month'] == latest_month]['csat'].mean(), 2)

# 2) Number of stores below CSAT target
TARGET = 4.0
by_store_latest = (
    df[df['month'] == latest_month]
    .groupby('store', as_index=False)['csat'].mean()
)
stores_below_target = int((by_store_latest['csat'] < TARGET).sum())

# 3) Total revenue (last 12 months) — replaces "total users"
total_revenue = round(last12['revenue_sar'].sum(), 2)

# 4) Return rate (latest month) — replaces quality/CSAT metric
latest = df[df['month'] == latest_month]
avg_return_rate = round((latest['returns_units'].sum() / latest['units_sold'].sum()) * 100, 2)

print(f'National average CSAT: {national_csat}')
print(f'Stores below target ({TARGET}): {stores_below_target}')
print(f'Total revenue (last 12 months): {total_revenue:,.2f} SAR')
print(f'Return rate (latest month): {avg_return_rate}%')

National average CSAT: 3.91
Stores below target (4.0): 6
Total revenue (last 12 months): 148,791,246.67 SAR
Return rate (latest month): 3.46%


In [96]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

fig = make_subplots(
    rows=3, cols=4,
    specs=[
        [{"type": "domain"}, {"type": "domain"}, {"type": "domain"}, {"type": "domain"}],
        [{"type": "xy", "colspan": 2}, None, {"type": "xy", "colspan": 2}, None],
        [{"type": "xy", "colspan": 4}, None, None, None],
    ],
    row_heights=[0.18, 0.42, 0.40],
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    subplot_titles=(
        None, None, None, None,  # Row of 4 KPI cards (no titles, the card itself is self-explanatory)
        "National revenue trend (last 12 months)",
        "Bottom 5 stores by CSAT (latest month)",
        "Revenue by channel over time",
    ),
)

fig.update_layout(height=650, width=1050, title="Empty layout — just the skeleton")
fig.show()

In [97]:
fig.add_trace(go.Indicator(
    mode="number", value=national_csat,
    number={"font": {"size": 32}},
    title={"text": "National Avg. CSAT"},
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number", value=stores_below_target,
    number={"font": {"size": 32, "color": "#D55E00" if stores_below_target else "#333"}},
    title={"text": "Stores Below Target"},
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="number", value=total_revenue,
    number={"valueformat": ",.0f", "suffix": " SAR", "font": {"size": 32}},
    title={"text": "Total Revenue (12mo)"},
), row=1, col=3)

fig.add_trace(go.Indicator(
    mode="number", value=avg_return_rate,
    number={"suffix": "%", "font": {"size": 32}},
    title={"text": "Return Rate"},
), row=1, col=4)

fig.update_layout(height=650, width=1050, title="Step 4.1 — KPI cards added")
fig.show()

In [98]:
national_trend = (
    df[df["month"] >= latest_month - pd.DateOffset(months=11)]
    .groupby("month", as_index=False)["revenue_sar"]
    .sum()
)

fig.add_trace(go.Scatter(
    x=national_trend["month"], y=national_trend["revenue_sar"],
    mode="lines+markers", line=dict(color="#1D9E75", width=2.5),
    showlegend=False,
), row=2, col=1)

fig.update_xaxes(title_text="Month", row=2, col=1)
fig.update_yaxes(title_text="Revenue (SAR)", row=2, col=1)

fig.update_layout(title="Step 4.2 — Trend line added")
fig.show()

In [99]:
bottom5 = by_store_latest.sort_values("csat").head(5)

fig.add_trace(go.Bar(
    x=bottom5["csat"], y=bottom5["store"], orientation="h",
    marker_color="#D55E00", showlegend=False,
), row=2, col=3)

fig.update_xaxes(title_text="CSAT Score", row=2, col=3)

fig.update_layout(title="Step 4.3 — Bottom 5 stores added")
fig.show()

In [100]:
by_channel = df.groupby("channel", as_index=False)["units_sold"].sum().sort_values("units_sold")

fig.add_trace(go.Bar(
    x=by_channel["units_sold"], y=by_channel["channel"], orientation="h",
    marker_color="#378ADD", showlegend=False,
), row=3, col=1)

fig.update_xaxes(title_text="Total units sold", row=3, col=1)

fig.update_layout(title="Step 4.4 — Channel breakdown added — dashboard complete")
fig.show()

In [101]:
fig.update_layout(
    title=dict(
        text="Retail Sales Dashboard — Store & Category Performance",
        font=dict(size=20, family="Arial"),
        x=0.5, xanchor="center",
    ),
    plot_bgcolor="white", paper_bgcolor="white",
    font=dict(family="Arial", size=12),
    height=680, width=1080,
    margin=dict(l=40, r=40, t=90, b=40),
)

for ann in fig.layout.annotations:
    ann.font = dict(size=13)

fig.show()

In [102]:
fig.write_html("retail_sales_dashboard.html", include_plotlyjs="cdn")
print("Saved:retail_sales_dashboard.html")
print("Open this file in any browser — no Python needed to view it.")

Saved:retail_sales_dashboard.html
Open this file in any browser — no Python needed to view it.
